# 📊 Caderno 3: Validação de Métricas e Engenharia de Dados (Issue #5)

Este caderno serve para validar manualmente se a matemática das funções não corrompeu os dados e analisar visualmente a volatilidade histórica.

## 1. Importação e Carga de Dados
Vamos carregar o dataset processado pelo nosso pipeline.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import plotly.express as px
import warnings

warnings.filterwarnings('ignore')

# Carrega o Dataset Analítico gerado pelo nosso pipeline
caminho_dados = '../data/processed/01_market_data_processed.csv'
df_processado = pd.read_csv(caminho_dados, sep=';', decimal=',')
df_processado['date'] = pd.to_datetime(df_processado['date'])

print(f"✅ Base de dados carregada! Total de registros: {len(df_processado)}")
display(df_processado.head())

## 2. Validação de Consistência (A Prova Real)

O retorno acumulado total deve bater com a variação direta do preço (Close_final / Close_inicial) - 1.

In [ ]:
ticker_teste = 'PETR4.SA'
df_teste = df_processado[df_processado['ticker'] == ticker_teste].copy()

# Pega o primeiro e último preço válido
preco_inicial = df_teste['close'].iloc[0]
preco_final = df_teste['close'].iloc[-1]

# Cálculo Cru
variacao_total = (preco_final / preco_inicial) - 1

# Nosso Cálculo Cumulativo (A coluna 'retorno_acumulado')
retorno_calculado = df_teste['retorno_acumulado'].iloc[-1]

print(f"--- 🧪 TESTE DE MATEMÁTICA FINANCEIRA: {ticker_teste} ---")
print(f"Variação de Preço Puro: {variacao_total:.4%}")
print(f"Variação pelo Retorno Acumulado: {retorno_calculado:.4%}")

if round(variacao_total, 4) == round(retorno_calculado, 4):
    print("✅ TESTE PASSOU: Os retornos diários estão matematicamente perfeitos!")
else:
    print("❌ ALERTA: Diferença matemática detectada!")

## 3. Análise Visual de Outliers e Volatilidade

Observando os momentos de pânico e crises através da volatilidade.

In [ ]:
fig = px.line(
    df_teste, 
    x='date', 
    y='volatilidade_21d', 
    title=f"Explosão de Outliers: Risco e Volatilidade de 21 Dias ({ticker_teste})",
    labels={'volatilidade_21d': 'Volatilidade (Desvio Padrão Anualizado)'},
    template='plotly_dark'
)

# Adiciona uma linha vermelha para identificar momentos de Pânico Extremo no mercado (Volatilidade > 60%)
fig.add_hline(y=0.60, line_dash="dash", line_color="red", annotation_text="Área de Pânico")

fig.show()